# 05. Annual model: external test and paired comparison

Scores the frozen annual model once on JECAM, each supplemental campaign and the combined pool, in the main testing layout. The paired comparison then scores it against each original window model on the parcels that model scored, with the parcel bootstrap of the external report.

These external parcels were already scored for the window models, so the comparison is retrospective evidence rather than a fresh held-out test.

In [ ]:
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from IPython.display import Markdown, display
from crop_pipeline.training import annual_workflow, paired_comparison

baseline_run = root / "pipeline_outputs/testing/run_2c43db2af7bc"  # original window models

In [ ]:
test = annual_workflow.test(root)
run = test.parent
for scope in json.loads((run / "reports/test_scopes.json").read_text()):
    folder = run / "reports" / ("" if scope == "jecam" else scope)
    scores = json.loads((folder / "test_scores.json").read_text())["annual"]
    print(f"{scope:40} parcels {sum(c['parcels'] for c in scores['coverage'].values()):4}  "
          f"macro-F1 {scores.get('macro_f1', 'unavailable')}")

In [ ]:
for revised in (False, True):
    output = root / "pipeline_outputs/reports" / f"annual_vs_windows_{run.name}{'_revised' if revised else ''}"
    if not output.exists():
        paired_comparison.report(run, "annual", baseline_run, output, revised=revised)
display(Markdown((root / "pipeline_outputs/reports" / f"annual_vs_windows_{run.name}" / "report.md").read_text()))